# 🛡️ Production Error Handling

**Handle errors gracefully in production LLM APIs**

---

## 📋 Overview

**What you'll learn:**
- Error types and handling
- Retry strategies
- Circuit breakers
- Fallback mechanisms
- Monitoring and alerting

**Time estimate:** ⏱️ 55 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from fastapi import FastAPI, HTTPException, Request, status
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel, Field, validator
from openai import OpenAI, AsyncOpenAI
from typing import Optional, Dict, Any
import time
import asyncio
import logging
import os

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Error Handling Matters

### Production Reality:

```
Things that WILL go wrong:
❌ OpenAI API is down
❌ Rate limits exceeded
❌ Network timeouts
❌ Invalid input from users
❌ Database connection lost
❌ Out of memory
❌ Malicious requests
```

### Bad Error Handling:

```python
# ❌ Don't do this!
@app.post("/chat")
async def chat(message: str):
    response = await client.chat.completions.create(...)
    return response

Problems:
- Crashes on any error
- No logging
- Poor user experience
- Hard to debug
```

### Good Error Handling:

```python
# ✅ Production-ready
@app.post("/chat")
async def chat(message: str):
    try:
        # Validate input
        validate_message(message)
        
        # Call with retry
        response = await call_with_retry(...)
        
        # Validate output
        validate_response(response)
        
        return response
    
    except ValidationError as e:
        logger.warning(f"Validation error: {e}")
        raise HTTPException(400, "Invalid input")
    
    except RateLimitError as e:
        logger.error(f"Rate limit: {e}")
        raise HTTPException(429, "Too many requests")
    
    except Exception as e:
        logger.exception(f"Unexpected error: {e}")
        raise HTTPException(500, "Internal error")
```

## 🎯 Error Types

In [ ]:
import openai
from enum import Enum

class ErrorType(str, Enum):
    """Common LLM API error types."""
    
    # Client errors (4xx) - User's fault
    VALIDATION = "validation_error"          # Invalid input
    AUTHENTICATION = "authentication_error"  # Bad API key
    RATE_LIMIT = "rate_limit_error"         # Too many requests
    QUOTA = "quota_exceeded"                # Out of credits
    
    # Server errors (5xx) - Our fault or OpenAI's
    TIMEOUT = "timeout_error"               # Request took too long
    API_ERROR = "api_error"                 # OpenAI API issue
    INTERNAL = "internal_error"             # Our code bug
    
    # Business logic
    CONTENT_FILTER = "content_filter"       # Unsafe content
    MAX_TOKENS = "max_tokens_exceeded"      # Response too long

# Error mapping
def classify_error(error: Exception) -> ErrorType:
    """Classify error type."""
    
    if isinstance(error, openai.AuthenticationError):
        return ErrorType.AUTHENTICATION
    
    elif isinstance(error, openai.RateLimitError):
        return ErrorType.RATE_LIMIT
    
    elif isinstance(error, openai.APITimeoutError):
        return ErrorType.TIMEOUT
    
    elif isinstance(error, openai.APIError):
        return ErrorType.API_ERROR
    
    elif isinstance(error, ValueError):
        return ErrorType.VALIDATION
    
    else:
        return ErrorType.INTERNAL

print("🎯 Error Classification")
print("\nError types:")
for error_type in ErrorType:
    print(f"  - {error_type.value}")

## 🔄 Retry Strategies

In [ ]:
import asyncio
import random
from typing import TypeVar, Callable

T = TypeVar('T')

async def retry_with_backoff(
    func: Callable[..., T],
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    exponential_base: float = 2.0,
    jitter: bool = True
) -> T:
    """Retry function with exponential backoff."""
    
    last_error = None
    
    for attempt in range(max_retries):
        try:
            return await func()
        
        except openai.RateLimitError as e:
            last_error = e
            
            if attempt == max_retries - 1:
                raise
            
            # Calculate delay
            delay = min(base_delay * (exponential_base ** attempt), max_delay)
            
            # Add jitter to prevent thundering herd
            if jitter:
                delay *= (0.5 + random.random() * 0.5)
            
            logger.warning(f"Rate limit hit, retrying in {delay:.2f}s (attempt {attempt + 1}/{max_retries})")
            await asyncio.sleep(delay)
        
        except openai.APITimeoutError as e:
            last_error = e
            
            if attempt == max_retries - 1:
                raise
            
            delay = base_delay * (exponential_base ** attempt)
            logger.warning(f"Timeout, retrying in {delay:.2f}s")
            await asyncio.sleep(delay)
        
        except openai.APIError as e:
            # Some API errors shouldn't be retried
            if "invalid" in str(e).lower():
                raise  # Don't retry validation errors
            
            last_error = e
            
            if attempt == max_retries - 1:
                raise
            
            delay = base_delay * (exponential_base ** attempt)
            logger.warning(f"API error, retrying in {delay:.2f}s")
            await asyncio.sleep(delay)
    
    raise last_error

# Example usage
async def call_openai_with_retry():
    """Call OpenAI with automatic retry."""
    
    async def make_request():
        return await client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": "Hello"}]
        )
    
    return await retry_with_backoff(
        make_request,
        max_retries=3,
        base_delay=1.0,
        exponential_base=2.0,
        jitter=True
    )

print("🔄 Retry Strategy")
print("\nBackoff schedule (base=1s, exponential=2):")
for i in range(5):
    delay = 1.0 * (2 ** i)
    print(f"  Attempt {i+1}: {delay:.1f}s")

## 🔌 Circuit Breaker

In [ ]:
from datetime import datetime, timedelta
from enum import Enum

class CircuitState(str, Enum):
    CLOSED = "closed"      # Normal operation
    OPEN = "open"          # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing if recovered

class CircuitBreaker:
    """Circuit breaker to prevent cascading failures."""
    
    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: int = 60,
        expected_exception: type = Exception
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.expected_exception = expected_exception
        
        self.failure_count = 0
        self.last_failure_time = None
        self.state = CircuitState.CLOSED
    
    async def call(self, func: Callable[..., T]) -> T:
        """Execute function with circuit breaker protection."""
        
        # Check if circuit should transition to half-open
        if self.state == CircuitState.OPEN:
            if self._should_attempt_reset():
                self.state = CircuitState.HALF_OPEN
                logger.info("Circuit breaker: OPEN → HALF_OPEN")
            else:
                raise Exception("Circuit breaker is OPEN")
        
        try:
            result = await func()
            self._on_success()
            return result
        
        except self.expected_exception as e:
            self._on_failure()
            raise
    
    def _should_attempt_reset(self) -> bool:
        """Check if enough time has passed to try again."""
        if self.last_failure_time is None:
            return True
        
        return (
            datetime.now() - self.last_failure_time
        ).total_seconds() >= self.recovery_timeout
    
    def _on_success(self):
        """Handle successful call."""
        if self.state == CircuitState.HALF_OPEN:
            logger.info("Circuit breaker: HALF_OPEN → CLOSED (recovered)")
        
        self.failure_count = 0
        self.state = CircuitState.CLOSED
    
    def _on_failure(self):
        """Handle failed call."""
        self.failure_count += 1
        self.last_failure_time = datetime.now()
        
        if self.failure_count >= self.failure_threshold:
            if self.state != CircuitState.OPEN:
                logger.error(
                    f"Circuit breaker: {self.state} → OPEN "
                    f"({self.failure_count} failures)"
                )
                self.state = CircuitState.OPEN
    
    def get_state(self) -> Dict[str, Any]:
        """Get circuit breaker state."""
        return {
            'state': self.state.value,
            'failure_count': self.failure_count,
            'last_failure': self.last_failure_time.isoformat() if self.last_failure_time else None
        }

# Global circuit breaker for OpenAI
openai_circuit = CircuitBreaker(
    failure_threshold=5,
    recovery_timeout=60,
    expected_exception=openai.APIError
)

print("🔌 Circuit Breaker")
print(f"\nStates:")
print(f"  CLOSED: Normal operation")
print(f"  OPEN: Failing, reject requests")
print(f"  HALF_OPEN: Testing recovery")
print(f"\nThreshold: {openai_circuit.failure_threshold} failures")
print(f"Recovery timeout: {openai_circuit.recovery_timeout}s")

## 🔀 Fallback Mechanisms

In [ ]:
from typing import List, Callable, Any

class FallbackChain:
    """Try multiple strategies with fallbacks."""
    
    def __init__(self):
        self.strategies: List[Callable] = []
    
    def add(self, strategy: Callable, name: str = None):
        """Add fallback strategy."""
        strategy._name = name or strategy.__name__
        self.strategies.append(strategy)
        return self
    
    async def execute(self, *args, **kwargs) -> Any:
        """Try strategies in order until one succeeds."""
        
        errors = []
        
        for i, strategy in enumerate(self.strategies):
            try:
                logger.info(f"Trying strategy {i+1}/{len(self.strategies)}: {strategy._name}")
                result = await strategy(*args, **kwargs)
                logger.info(f"Strategy {strategy._name} succeeded")
                return result
            
            except Exception as e:
                logger.warning(f"Strategy {strategy._name} failed: {e}")
                errors.append((strategy._name, e))
                
                if i == len(self.strategies) - 1:
                    # All strategies failed
                    raise Exception(
                        f"All {len(self.strategies)} strategies failed: {errors}"
                    )

# Example: LLM with fallbacks
async def gpt4_strategy(prompt: str) -> str:
    """Try GPT-4 first (best quality)."""
    response = await client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        timeout=10
    )
    return response.choices[0].message.content

async def gpt35_strategy(prompt: str) -> str:
    """Fallback to GPT-3.5 (faster, cheaper)."""
    response = await client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        timeout=5
    )
    return response.choices[0].message.content

async def cached_strategy(prompt: str) -> str:
    """Try cached response."""
    # Check cache (simplified)
    cached = None  # cache.get(prompt)
    if cached:
        return cached
    raise Exception("Not in cache")

async def default_response(prompt: str) -> str:
    """Last resort: generic response."""
    return "I'm having trouble processing your request. Please try again later."

# Build fallback chain
llm_chain = FallbackChain()
llm_chain.add(cached_strategy, "Cache")
llm_chain.add(gpt4_strategy, "GPT-4")
llm_chain.add(gpt35_strategy, "GPT-3.5")
llm_chain.add(default_response, "Default")

print("🔀 Fallback Chain")
print("\nStrategy order:")
for i, strategy in enumerate(llm_chain.strategies, 1):
    print(f"  {i}. {strategy._name}")

## 🏗️ Complete Error Handling System

In [ ]:
print("""
# Complete production error handling

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.exceptions import RequestValidationError
from pydantic import BaseModel
import logging
import time

app = FastAPI(title="Production LLM API")
logger = logging.getLogger(__name__)

# Global error handlers
@app.exception_handler(RequestValidationError)
async def validation_exception_handler(request: Request, exc: RequestValidationError):
    \"\"\"Handle validation errors.\"\"\" 
    logger.warning(f"Validation error: {exc.errors()}")
    return JSONResponse(
        status_code=400,
        content={
            "error": "Validation error",
            "details": exc.errors()
        }
    )

@app.exception_handler(openai.RateLimitError)
async def rate_limit_handler(request: Request, exc: openai.RateLimitError):
    \"\"\"Handle rate limits.\"\"\" 
    logger.error(f"Rate limit exceeded: {exc}")
    return JSONResponse(
        status_code=429,
        content={
            "error": "Rate limit exceeded",
            "message": "Too many requests. Please try again later.",
            "retry_after": 60
        },
        headers={"Retry-After": "60"}
    )

@app.exception_handler(openai.APIError)
async def api_error_handler(request: Request, exc: openai.APIError):
    \"\"\"Handle OpenAI API errors.\"\"\" 
    logger.error(f"OpenAI API error: {exc}", exc_info=True)
    return JSONResponse(
        status_code=502,
        content={
            "error": "LLM service unavailable",
            "message": "Please try again later"
        }
    )

@app.exception_handler(Exception)
async def general_exception_handler(request: Request, exc: Exception):
    \"\"\"Handle all other errors.\"\"\" 
    logger.exception(f"Unexpected error: {exc}")
    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal server error",
            "message": "Something went wrong. Please try again."
        }
    )

# Request/response logging middleware
@app.middleware("http")
async def log_requests(request: Request, call_next):
    \"\"\"Log all requests and responses.\"\"\" 
    start_time = time.time()
    request_id = str(uuid.uuid4())
    
    logger.info(f"[{request_id}] {request.method} {request.url.path}")
    
    try:
        response = await call_next(request)
        
        duration = time.time() - start_time
        logger.info(
            f"[{request_id}] {response.status_code} "
            f"({duration*1000:.2f}ms)"
        )
        
        response.headers["X-Request-ID"] = request_id
        return response
    
    except Exception as e:
        duration = time.time() - start_time
        logger.error(
            f"[{request_id}] Error after {duration*1000:.2f}ms: {e}",
            exc_info=True
        )
        raise

# Protected endpoint
class ChatRequest(BaseModel):
    message: str

@app.post("/api/chat")
async def chat(request: ChatRequest):
    \"\"\"Chat endpoint with full error handling.\"\"\" 
    
    try:
        # Call with retry and circuit breaker
        async def make_request():
            return await client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": request.message}]
            )
        
        # Try with fallback chain
        response = await llm_chain.execute(request.message)
        
        return {
            "response": response,
            "status": "success"
        }
    
    except Exception as e:
        # Let global handlers deal with it
        raise

@app.get("/health")
async def health():
    \"\"\"Health check with circuit breaker status.\"\"\" 
    return {
        "status": "healthy",
        "circuit_breaker": openai_circuit.get_state()
    }
""")

## ✅ Summary

### Error Handling Checklist:

**✅ Classify Errors**
```python
- Client errors (4xx): User's fault
- Server errors (5xx): Our fault
- Retryable vs non-retryable
```

**✅ Retry Logic**
```python
- Exponential backoff
- Jitter to prevent thundering herd
- Max retries limit
- Only retry retryable errors
```

**✅ Circuit Breaker**
```python
- Prevent cascading failures
- Fail fast when service is down
- Auto-recovery with half-open state
```

**✅ Fallbacks**
```python
- Try cache first
- Fallback to cheaper model
- Default response as last resort
```

**✅ Logging**
```python
- Log all errors
- Include context (request ID, user, etc.)
- Use structured logging
- Different log levels
```

### Best Practices:

**1. Use Proper HTTP Status Codes**
```python
400 Bad Request       # Invalid input
401 Unauthorized      # Bad auth
429 Too Many Requests # Rate limit
500 Internal Error    # Our bug
502 Bad Gateway       # Upstream service down
503 Service Unavailable # Maintenance
```

**2. Return Helpful Error Messages**
```python
# ❌ Bad
{"error": "Error"}

# ✅ Good
{
    "error": "Validation error",
    "message": "Message is required",
    "details": [{"field": "message", "error": "required"}]
}
```

**3. Don't Expose Internals**
```python
# ❌ Bad - exposes internal error
raise HTTPException(500, detail=str(e))

# ✅ Good - generic message
logger.exception(f"Error: {e}")
raise HTTPException(500, detail="Internal server error")
```

**4. Add Request IDs**
```python
request_id = str(uuid.uuid4())
logger.info(f"[{request_id}] Processing request")
response.headers["X-Request-ID"] = request_id
```

**5. Set Timeouts**
```python
response = await asyncio.wait_for(
    client.chat.completions.create(...),
    timeout=30.0
)
```

### Production Patterns:

**Retry + Circuit Breaker + Fallback:**
```python
try:
    # Try with circuit breaker
    result = await openai_circuit.call(
        lambda: retry_with_backoff(make_request)
    )
except Exception:
    # Fallback to cache or default
    result = await fallback_chain.execute()
```

**Global Error Handlers:**
```python
@app.exception_handler(Exception)
async def handle_all_errors(request, exc):
    logger.exception(f"Error: {exc}")
    return JSONResponse(
        status_code=500,
        content={"error": "Internal error"}
    )
```

**Health Checks:**
```python
@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "circuit_breaker": circuit.get_state(),
        "timestamp": datetime.now().isoformat()
    }
```

### Next Steps:

You've completed the Production APIs basics! Next:
- Authentication & authorization
- Rate limiting
- Caching strategies
- Observability & monitoring

### Next: `09_caching/01_response_caching.ipynb`